# Notebook 1: Data Exploration — Mozilla Common Voice Arabic

This notebook explores the Arabic speech dataset we use for training.

**Contents:**
1. Load dataset
2. Inspect samples
3. Audio statistics (duration, sample rate)
4. Visualize waveforms and mel spectrograms
5. Transcript statistics (length, character frequency)
6. Dataset splits summary

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from collections import Counter
from datasets import load_dataset, Audio
from utils.audio_utils import extract_mel_spectrogram, normalize_audio
from utils.visualization import plot_waveform, plot_mel_spectrogram

print('Imports OK')

## 1. Load Mozilla Common Voice Arabic

In [ ]:
# Load a small subset for exploration
dataset = load_dataset(
    'mozilla-foundation/common_voice_13_0',
    'ar',
    split='train',
    trust_remote_code=True,
)
dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))

print(f'Training samples: {len(dataset)}')
print(f'Columns: {dataset.column_names}')
print(f'\nFirst sample keys: {dataset[0].keys()}')

In [ ]:
# Show first few samples
for i in range(3):
    s = dataset[i]
    dur = len(s['audio']['array']) / s['audio']['sampling_rate']
    print(f"Sample {i}: '{s['sentence']}' — {dur:.2f}s")

## 2. Audio Duration Statistics

In [ ]:
# Compute durations for first 1000 samples
n_inspect = min(1000, len(dataset))
durations = []
for i in range(n_inspect):
    s = dataset[i]
    dur = len(s['audio']['array']) / s['audio']['sampling_rate']
    durations.append(dur)

durations = np.array(durations)
print(f'Duration stats (n={n_inspect}):')
print(f'  Mean:   {durations.mean():.2f}s')
print(f'  Median: {np.median(durations):.2f}s')
print(f'  Min:    {durations.min():.2f}s')
print(f'  Max:    {durations.max():.2f}s')
print(f'  Std:    {durations.std():.2f}s')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(durations.mean(), color='red', linestyle='--', label=f'Mean: {durations.mean():.2f}s')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.set_title('Audio Duration Distribution — Common Voice Arabic')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Waveform and Mel Spectrogram Visualization

In [ ]:
# Pick a sample and visualize
sample = dataset[5]
waveform = sample['audio']['array'].astype(np.float32)
sr = sample['audio']['sampling_rate']
transcript = sample['sentence']

print(f'Transcript: {transcript}')
print(f'Duration: {len(waveform)/sr:.2f}s | Sample rate: {sr}Hz')

plot_waveform(waveform, sr, title=f'Waveform — "{transcript[:30]}..."')

In [ ]:
# Mel spectrogram
waveform_norm = normalize_audio(waveform)
mel = extract_mel_spectrogram(waveform_norm, sample_rate=sr)

print(f'Mel spectrogram shape: {mel.shape}  (n_mels × time_frames)')
plot_mel_spectrogram(mel, sr, title=f'Log-Mel Spectrogram — "{transcript[:30]}..."')

## 4. Transcript Statistics

In [ ]:
transcripts = [dataset[i]['sentence'] for i in range(n_inspect)]
word_counts = [len(t.split()) for t in transcripts]
char_counts = [len(t) for t in transcripts]

print(f'Word count — Mean: {np.mean(word_counts):.1f}, Max: {max(word_counts)}')
print(f'Char count — Mean: {np.mean(char_counts):.1f}, Max: {max(char_counts)}')

# All characters in corpus
all_chars = Counter(''.join(transcripts))
print(f'\nVocabulary size (unique chars): {len(all_chars)}')

# Top 20 chars
top_chars = all_chars.most_common(20)
chars, freqs = zip(*top_chars)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(chars)), freqs, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(chars)))
ax.set_xticklabels(chars, fontsize=12)
ax.set_ylabel('Frequency')
ax.set_title('Top 20 Most Frequent Characters in Arabic Transcripts')
plt.tight_layout()
plt.show()

## 5. Dataset Splits Summary

In [ ]:
for split in ['train', 'validation', 'test']:
    ds = load_dataset('mozilla-foundation/common_voice_13_0', 'ar', split=split, trust_remote_code=True)
    print(f'{split:12}: {len(ds):6,} samples')